In [9]:
import numpy as np
import pandas as pd

dfv1 = pd.read_csv(r'E:\Code\FireCast_Experimental\firecast_data_readymodelling_final_V3.csv')
dfv1['time'] = pd.to_datetime(dfv1['time'])
dfv1 = dfv1.sort_values('time').reset_index(drop=True)

print(f"Loaded {len(dfv1)} rows")
print(f"Columns: {dfv1.columns.tolist()}")


# Temporal Features
numeric_cols = dfv1.select_dtypes(include=[np.number]).columns.drop(['label'])
for col in numeric_cols:
    if col not in ['label']:
        dfv1[f'{col}_lag3'] = dfv1[col].shift(3)
        dfv1[f'{col}_roll7'] = dfv1[col].rolling(window=7, min_periods=1).mean()


# Fire Indices
def add_fire_indices(df):
    df = df.copy()

    # 1. Vapor Pressure Deficit (VPD) - better drought indicator than humidity alone
    # Approximation: VPD (0.6108 * exp((17.27 * temp)/(temp + 237.3))) * (1 - RH/100)

    df['vpd'] = df['temp_max'] - df['dewpoint']  # Simple proxy; higher = drier air

    # 2. Wind magnitude (you have components, but also raw speed)
    df['wind_mag'] = np.sqrt(df['wind_u']**2 + df['wind_v']**2)

    # 3. Drought Stress Index: cumulative dry days
    df['dry_day'] = (df['precip'] < 1.0).astype(int)  # <1mm = dry
    df['dry_spell_7'] = df['dry_day'].rolling(7, min_periods=1).sum()
    df['dry_spell_30'] = df['dry_day'].rolling(30, min_periods=1).sum()

    # 4. Fuel dryness proxy: combine VPD + NDVI (dry vegetation = high VPD + low NDVI)
    df['fuel_dryness'] = df['vpd'] * (1 - df['ndvi'])  # Higher = more flammable

    # 5. Fire weather type: hot + dry + windy
    df['hot_threshold'] = (df['temp_max'] > df['temp_max'].quantile(0.75)).astype(int)
    df['dry_threshold'] = (df['vpd'] > df['vpd'].quantile(0.75)).astype(int)
    df['windy_threshold'] = (df['wind_speed'] > df['wind_speed'].quantile(0.75)).astype(int)
    df['extreme_fire_weather'] = df['hot_threshold'] + df['dry_threshold'] + df['windy_threshold']

    return df

# Spectral Indices
def add_spectral_indices(df):
    df = df.copy()

    # NDWI (Normalized Difference Water Index) - moisture in vegetation
    # Uses B3 (green) and B8 (NIR) or B11 (SWIR) B11 is better for moisture
    df['ndwi'] = (df['B3'] - df['B11']) / (df['B3'] + df['B11'] + 1e-8)

    # NBR (Normalized Burn Ratio) - pre-fire fuel load
    # NBR = (NIR - SWIR) / (NIR + SWIR) use B8 (NIR) and B12 (SWIR)
    df['nbr'] = (df['B8'] - df['B12']) / (df['B8'] + df['B12'] + 1e-8)

    # EVI (Enhanced Vegetation Index) - better than NDVI in high biomass
    # EVI = 2.5 * (NIR - Red) / (NIR + 6*Red - 7.5*Blue + 1)
    df['evi'] = 2.5 * (df['B8'] - df['B4']) / (df['B8'] + 6*df['B4'] - 7.5*df['B2'] + 1 + 1e-8)

    return df

dfv1 = add_fire_indices(dfv1)
dfv1 = add_spectral_indices(dfv1)

# Cyclical time features
dfv1['month_sin'] = np.sin(2 * np.pi * dfv1['month'] / 12)
dfv1['month_cos'] = np.cos(2 * np.pi * dfv1['month'] / 12)
dfv1['doy'] = dfv1['time'].dt.dayofyear
dfv1['doy_sin'] = np.sin(2 * np.pi * dfv1['doy'] / 365)
dfv1['doy_cos'] = np.cos(2 * np.pi * dfv1['doy'] / 365)

# Trend Based features
# 1. Rate of change (how fast conditions are deteriorating)
dfv1['temp_change_3d'] = dfv1['temp_max'].diff(3)
dfv1['wind_change_3d'] = dfv1['wind_speed'].diff(3)

# 2. Cumulative dryness (not just count of dry days)
dfv1['precip_deficit_7d'] = (dfv1['precip'].rolling(7).mean() - dfv1['precip'].mean()).clip(lower=0)

# 3. Fire weather persistence
dfv1['extreme_days_5d'] = (dfv1['extreme_fire_weather'].rolling(5).sum() > 0).astype(int)

dfv1.replace([np.inf, -np.inf], np.nan, inplace=True)
dfv1 = dfv1.dropna().reset_index(drop=True)

print(f"Final dataset has {len(dfv1)} rows and {len(dfv1.columns)} columns after feature engineering.")
print(f"Label distribution:\n{dfv1['label'].value_counts(normalize=True)}")

dfv1.to_csv('../processed_firecast_features_v3.csv', index=False)

Loaded 56222 rows
Columns: ['B11', 'B12', 'B2', 'B3', 'B4', 'B8', 'dewpoint', 'elevation', 'land_cover', 'lat', 'lon', 'ndvi', 'precip', 'temp_max', 'time', 'wind_speed', 'wind_u', 'wind_v', 'month', 'day', 'label']
Final dataset has 56216 rows and 81 columns after feature engineering.
Label distribution:
label
1    0.549505
0    0.450495
Name: proportion, dtype: float64
